# S09. Tipos de Amostras
## Types of Samples

[◀ Anterior](S08_Tipos_de_Estudo.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S10_Tipos_de_Dados.ipynb)

## 🎯 Objetivos
- Conhecer os principais métodos de amostragem probabilística e não-probabilística
- Compreender quando aplicar cada tipo de amostragem
- Implementar diferentes técnicas de amostragem em Python
- Avaliar vantagens e desvantagens de cada método

## 📝 Introdução

A forma como selecionamos os elementos de uma amostra é tão importante quanto o tamanho dela. Um método de amostragem inadequado pode introduzir vieses que invalidam qualquer conclusão. Existem duas grandes famílias de métodos: **amostragem probabilística** (cada elemento tem probabilidade conhecida de ser selecionado) e **amostragem não-probabilística** (a seleção depende de critérios subjetivos ou de conveniência).

## 📚 Explicação Detalhada

### 1. Amostragem Aleatória Simples

Na **amostragem aleatória simples** (simple random sampling), cada elemento da população tem exatamente a mesma probabilidade de ser escolhido. É o método mais básico e o padrão ouro da amostragem probabilística.

**Como implementar:**
- Numerar todos os elementos da população de 1 a N
- Sortear n números aleatórios sem reposição
- Selecionar os elementos correspondentes

**Vantagens:**
- Simples de entender e implementar
- Livre de viés de seleção
- Base teórica sólida para inferência

**Desvantagens:**
- Exige lista completa da população (nem sempre disponível)
- Pode ser cara se a população for geograficamente dispersa
- Pode não capturar subgrupos pequenos

In [ ]:
# Amostragem Aleatória Simples
import numpy as np
import pandas as pd

np.random.seed(42)

# População: 10.000 funcionários de uma empresa
funcionarios = pd.DataFrame({
    'id': range(1, 10001),
    'departamento': np.random.choice(
        ['Vendas', 'TI', 'RH', 'Financeiro', 'Marketing', 'Operações'],
        10000, p=[0.25, 0.20, 0.10, 0.15, 0.10, 0.20]
    ),
    'salario': np.random.lognormal(8.5, 0.4, 10000),
    'tempo_casa': np.random.randint(1, 360, 10000)  # meses
})

# Amostragem aleatória simples (AAS) de 500 funcionários
# Método 1: usando numpy
indices_sorteados = np.random.choice(10000, size=500, replace=False)
aas_numpy = funcionarios.iloc[indices_sorteados]

# Método 2: usando pandas .sample()
aas_pandas = funcionarios.sample(n=500, random_state=42)

print('Amostragem Aleatória Simples — 500 funcionários:')
print(aas_pandas[['id', 'departamento', 'salario']].head(10))
print(f'\nProporção por departamento:')
print(aas_pandas['departamento'].value_counts(normalize=True).sort_index())

### 2. Amostragem Estratificada

Na **amostragem estratificada** (stratified sampling), a população é dividida em subgrupos homogêneos chamados **estratos** (strata), e uma amostra é extraída de cada estrato — proporcional ou uniformemente.

**Quando usar:**
- A população tem subgrupos claramente definidos
- Os subgrupos diferem na variável de interesse
- Queremos garantir representação de todos os subgrupos

**Tipos:**
- **Proporcional**: o tamanho da amostra em cada estrato é proporcional ao tamanho do estrato na população
- **Uniforme**: mesmo número de elementos por estrato (útil quando estratos pequenos são importantes)

In [ ]:
# Amostragem Estratificada
np.random.seed(42)

# População com departamentos de tamanhos diferentes
departamentos = ['Vendas', 'TI', 'RH', 'Financeiro', 'Marketing', 'Operações']
proporcoes = [0.25, 0.20, 0.10, 0.15, 0.10, 0.20]
tamanhos = [int(10000 * p) for p in proporcoes]

# Criando a população estratificada
populacao_estratos = {}
for dept, n in zip(departamentos, tamanhos):
    populacao_estratos[dept] = pd.DataFrame({
        'id': range(1, n+1),
        'departamento': dept,
        'salario': np.random.lognormal(8.2 + 0.1 * departamentos.index(dept), 0.3, n)
    })

populacao = pd.concat(populacao_estratos.values()).reset_index(drop=True)

# Amostragem estratificada proporcional
total_amostra = 500
amostra_estratos = []
for dept in departamentos:
    n_estrato = int(total_amostra * proporcoes[departamentos.index(dept)])
    df_estrato = populacao_estratos[dept]
    amostra_est = df_estrato.sample(n=n_estrato, random_state=42)
    amostra_estratos.append(amostra_est)

amostra_estratificada = pd.concat(amostra_estratos).reset_index(drop=True)

print('Amostragem Estratificada Proporcional:')
print(f'Total: {len(amostra_estratificada)} funcionários')
print(f'\nComparação de proporções:')
for dept in departamentos:
    pop_pct = proporcoes[departamentos.index(dept)]
    amostra_pct = len(amostra_estratificada[amostra_estratificada['departamento'] == dept]) / total_amostra
    print(f'{dept:15s}: pop={pop_pct:.2f}, amostra={amostra_pct:.2f}')

# Comparação salarial
salario_real = populacao['salario'].mean()
salario_estrat = amostra_estratificada['salario'].mean()
print(f'\nSalário médio real: R$ {salario_real:.2f}')
print(f'Salário médio amostra estratificada: R$ {salario_estrat:.2f}')
print(f'Erro: R$ {abs(salario_real - salario_estrat):.2f}')

### 3. Amostragem por Conglomerados (Cluster)

Na **amostragem por conglomerados** (cluster sampling), a população é dividida em grupos naturais (conglomerados/clusters), sortearmos alguns conglomerados e incluímos **todos** os elementos dos conglomerados sorteados.

**Diferença para estratificada:**
- **Estratos**: você amostra de **todos** os estratos
- **Clusters**: você sorteia **alguns** clusters e inclui todos os seus elementos

**Quando usar:**
- População naturalmente agrupada (escolas, bairros, cidades)
- Lista completa da população não está disponível, mas lista de clusters sim
- Reduzir custos de deslocamento (geograficamente disperso)

In [ ]:
# Amostragem por Conglomerados (Cluster)
np.random.seed(42)

# População: 50 escolas, cada uma com 200-400 alunos
n_escolas = 50
escolas = {}
for i in range(1, n_escolas + 1):
    n_alunos = np.random.randint(200, 400)
    escolas[i] = pd.DataFrame({
        'escola': i,
        'aluno_id': range(1, n_alunos + 1),
        'nota': np.random.normal(65 + np.random.uniform(-10, 10), 12, n_alunos)
    })

# Amostragem por cluster: sorteamos 5 escolas e incluímos TODOS os alunos
cluster_sorteados = np.random.choice(list(escolas.keys()), size=5, replace=False)

amostra_cluster = pd.concat([escolas[e] for e in cluster_sorteados]).reset_index(drop=True)

print('Amostragem por Conglomerados (Cluster):')
print(f'Total de escolas na população: {n_escolas}')
print(f'Escolas sorteadas: {sorted(cluster_sorteados)}')
print(f'Total de alunos na amostra: {len(amostra_cluster)}')
print(f'\nAlunos por escola sorteada:')
for e in cluster_sorteados:
    print(f'  Escola {e}: {len(escolas[e])} alunos')

# Média populacional vs amostral
todos_alunos = pd.concat(escolas.values())
nota_real = todos_alunos['nota'].mean()
nota_cluster = amostra_cluster['nota'].mean()
print(f'\nNota média real: {nota_real:.2f}')
print(f'Nota média amostra cluster: {nota_cluster:.2f}')
print(f'Erro: {abs(nota_real - nota_cluster):.2f}')

### 4. Outros Métodos de Amostragem

**Amostragem Sistemática:**
Seleciona elementos em intervalos regulares (k = N/n). Ex: a cada 10º cliente que entra na loja. Simples, mas pode ser viesada se houver periodicidade oculta nos dados.

**Amostragem de Conveniência:**
Seleciona elementos por facilidade de acesso. Ex: entrevistar pessoas na rua. Rápida e barata, mas altamente sujeita a viés.

**Amostragem Snowball (Bola de Neve):**
Participantes indicam outros participantes. Ideal para populações de difícil acesso (usuários de drogas, profissionais nichados).

**Amostragem por Cotas:**
Define cotas para subgrupos (ex: 50% homens, 50% mulheres), mas a seleção dentro da cota é por conveniência. Híbrida entre probabilística e não-probabilística.

In [ ]:
# Comparação de métodos de amostragem
np.random.seed(42)

# População
populacao = pd.DataFrame({
    'id': range(1, 5001),
    'regiao': np.random.choice(['Norte', 'Sul', 'Leste', 'Oeste'], 5000),
    'renda': np.random.lognormal(8, 0.5, 5000)
})

renda_real = populacao['renda'].mean()
print(f'Renda média populacional real: R$ {renda_real:.2f}')
print()

# 1. Amostragem Aleatória Simples (AAS)
aas = populacao.sample(n=200, random_state=1)
erro_aas = abs(renda_real - aas['renda'].mean())
print(f'1. AAS:                R$ {aas["renda"].mean():.2f}  erro: R$ {erro_aas:.2f}')

# 2. Amostragem Sistemática (k = 5000/200 = 25)
k = 25
inicio = np.random.randint(0, k)
sistematica = populacao.iloc[inicio::k].head(200)
erro_sis = abs(renda_real - sistematica['renda'].mean())
print(f'2. Sistemática (k={k}): R$ {sistematica["renda"].mean():.2f}  erro: R$ {erro_sis:.2f}')

# 3. Amostragem de Conveniência (primeiros 200)
conveniencia = populacao.head(200)
erro_conv = abs(renda_real - conveniencia['renda'].mean())
print(f'3. Conveniência:       R$ {conveniencia["renda"].mean():.2f}  erro: R$ {erro_conv:.2f}')

# 4. Amostragem Estratificada por Região
amostra_est = populacao.groupby('regiao', group_keys=False).apply(
    lambda x: x.sample(n=int(200 * len(x) / len(populacao)), random_state=1)
)
erro_est = abs(renda_real - amostra_est['renda'].mean())
print(f'4. Estratificada:      R$ {amostra_est["renda"].mean():.2f}  erro: R$ {erro_est:.2f}')
print()
print('AAS e estratificada geralmente têm menor erro por serem probabilísticas.')

## 💻 Exemplos Práticos

In [ ]:
# Exemplo prático: combinando métodos de amostragem
import matplotlib.pyplot as plt

np.random.seed(42)

# Pesquisa de satisfação em uma universidade com 15.000 alunos
# Estratificamos por curso, depois AAS dentro de cada estrato

cursos = ['Engenharia', 'Medicina', 'Direito', 'Arquitetura', 'Computação']
alunos_por_curso = [4000, 1500, 3000, 2000, 4500]

pop_cursos = {}
for curso, n in zip(cursos, alunos_por_curso):
    pop_cursos[curso] = pd.DataFrame({
        'curso': curso,
        'satisfacao': np.random.uniform(3, 10, n),
        'periodo': np.random.choice(['Manhã', 'Tarde', 'Noite'], n)
    })

pop_total = pd.concat(pop_cursos.values()).reset_index(drop=True)

# Amostragem estratificada proporcional (n = 600)
total_amostra = 600
amostra_final = []

for curso in cursos:
    df_curso = pop_cursos[curso]
    n_curso = int(total_amostra * len(df_curso) / len(pop_total))
    # Dentro de cada curso, AAS
    amostra_curso = df_curso.sample(n=n_curso, random_state=42)
    amostra_final.append(amostra_curso)

pesquisa = pd.concat(amostra_final).reset_index(drop=True)

print('Pesquisa de Satisfação — Amostra Estratificada:')
print(f'Total de alunos: {len(pop_total)}')
print(f'Tamanho da amostra: {len(pesquisa)}')
print(f'\nSatisfação média geral: {pesquisa["satisfacao"].mean():.2f}')
print(f'Satisfação média real: {pop_total["satisfacao"].mean():.2f}')
print()

# Média por curso
for curso in cursos:
    real = pop_cursos[curso]['satisfacao'].mean()
    est = pesquisa[pesquisa['curso'] == curso]['satisfacao'].mean()
    print(f'{curso:15s}: real={real:.2f}, estimado={est:.2f}, erro={abs(real-est):.2f}')

## ✅ Exercícios Resolvidos

**Exercício 1:** Um pesquisador quer estudar a saúde bucal de crianças em uma cidade com 50 escolas. Qual método de amostragem seria mais adequado e por quê?

In [ ]:
# Solução:
# Amostragem por conglomerados (clusters) é a mais adequada porque:
# 1. As escolas são clusters naturais
# 2. Sortear escolas é mais barato que sortear crianças individualmente
# 3. Não precisa de lista de todas as crianças, apenas das escolas
#
# Passos:
# 1. Listar todas as 50 escolas (clusters)
# 2. Sortear 8-10 escolas aleatoriamente
# 3. Examinar TODAS as crianças das escolas sorteadas
#
# Alternativa: amostragem estratificada por região da cidade,
# se houver diferenças regionais relevantes.

print('Método recomendado: amostragem por conglomerados (escolas como clusters).')
print('Vantagens: menor custo operacional, sem necessidade de censo de alunos.')

## 📋 Exercícios Propostos

| Nível | Exercício |
|-------|-----------|
| 🟢 Fácil | 1. Explique a diferença entre amostragem probabilística e não-probabilística. |
| 🟡 Médio | 2. Uma pesquisa eleitoral quer garantir representação proporcional de todas as faixas etárias. Que método usar? Implemente. |
| 🔴 Difícil | 3. Simule uma população com 100.000 elementos e compare a precisão de 4 métodos de amostragem repetindo cada um 500 vezes. Qual método produz o menor erro médio? |

In [ ]:
# 🟢 Exercício 1
# Amostragem probabilística: cada elemento tem probabilidade conhecida e não-nula
#   de ser selecionado. Permite inferência estatística.
# Ex: AAS, estratificada, cluster, sistemática.
#
# Amostragem não-probabilística: seleção por critérios subjetivos.
#   Não permite generalização com base estatística.
# Ex: conveniência, snowball, por cotas.

## 🏆 Desafios

1. Crie uma função genérica que aplique qualquer método de amostragem dado um DataFrame e os parâmetros
2. Implemente uma simulação de Monte Carlo para comparar a eficiência de AAS vs estratificada em diferentes cenários
3. Pesquise sobre amostragem adaptativa e crie um exemplo prático

## 📌 Resumo
- **AAS**: cada elemento tem igual probabilidade — padrão ouro
- **Estratificada**: divide em estratos, amostra de todos — garante representatividade
- **Cluster**: sorteia grupos, inclui todos — econômica para populações dispersas
- **Sistemática**: intervalos regulares — simples, cuidado com periodicidade
- **Conveniência**: fácil mas viesada — usar apenas em estudos exploratórios
- **Snowball**: indicação em cadeia — para populações de difícil acesso

## 💡 Curiosidades
- O IBGE usa amostragem estratificada por setor censitário para a Pesquisa Nacional por Amostra de Domicílios (PNAD)
- A amostragem snowball é usada para estudar populações como profissionais do sexo, usuários de drogas e portadores de doenças raras
- A amostragem sistemática quase causou um desastre na Guerra do Vietnã: soldados usavam intervalos regulares para patrulhar, e inimigos aprenderam o padrão

## ✅ Boas Práticas
- Prefira métodos probabilísticos sempre que possível
- Documente detalhadamente o método de amostragem utilizado
- Para amostragem estratificada, defina os estratos com base em variáveis relevantes
- Em amostragem sistemática, verifique se não há periodicidade oculta
- Reporte a taxa de resposta e possíveis vieses de não-resposta

## ⚠️ Erros Comuns

| Erro | Como Evitar |
|------|-------------|
| Usar amostra de conveniência e generalizar | Apenas métodos probabilísticos permitem inferência |
| Confundir estratificada com cluster | Estratos: todos incluídos; Clusters: apenas alguns sorteados |
| Ignorar a taxa de não-resposta | Calcular e reportar a taxa de resposta efetiva |
| Amostragem sistemática com periodicidade | Verificar ordenação dos dados antes de aplicar |

## 📖 Referências

- [W3Schools Statistics — Sampling Methods](https://www.w3schools.com/statistics/statistics_sampling_methods.php)
- [IBGE — PNAD](https://www.ibge.gov.br/estatisticas/sociais/trabalho/9171-pesquisa-nacional-por-amostra-de-domicilios-continua-mensal.html)
- [Wikipedia — Sampling](https://en.wikipedia.org/wiki/Sampling_(statistics))

[◀ Anterior](S08_Tipos_de_Estudo.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S10_Tipos_de_Dados.ipynb)

---
🧠 **Quer uma abordagem mais intuitiva?**
Visite o [companheiro de analogia: Tipos de Amostras — O Buffet de Casamento](S53_Tipos_de_Amostras_Analogias.ipynb)
para aprender este conteudo com uma historia do dia a dia.
